# 7.13 後續改動要和什麼比？

# 第 7 章：從續寫文字到回答問題

前置：第4章的因果模型與第1章的資料對齊。先用一筆不截斷對話，再回讀padding。完整render在tiny_perceptron/data.py；下列程式每節都新建資料，不依賴上一Notebook變量。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩種遮罩：看得見，不一定直接考它**

X 的 assistant 格預測 A；PAD 不當答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/masks.svg")))

**先想一想：**JSON能解析就代表答案內容對嗎？

先保存固定任務與判讀規則，後續風格、安全、量化纔有共同基準。把格式符合、內容正確、適當不確定分別記錄。

**把直覺寫成數學：**一個向量指標比單一總分保留更多行爲差異。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
import json

reply = '{"answer": 3}'
parsed = json.loads(reply)
scores = {"JSON有效": isinstance(parsed, dict), "1+1內容正確": parsed.get("answer") == 2}
print(scores)
assert scores["JSON有效"] and not scores["1+1內容正確"]

**如何讀結果：**固定測試內容以後不應反覆用來微調；另保留final test。

**只改一件事：**只把answer改成2，觀察兩個面向分別變化。
